# Social Network Extraction on CSD3

Run this notebook on an A100 node via `salloc`. See `README.md` for full setup instructions.

**Prerequisites:**
- `llm` kernel selected (venv with vllm + largeliterarymodels)
- Passages exported to `~/rds/hpc-work/texts/` as JSONL
- Model weights pre-downloaded

In [ ]:
import os, subprocess, time, urllib.request, glob, json

RDS = os.path.expanduser('~/rds/hpc-work')
TEXT_DIR = os.path.join(RDS, 'texts')
OUTPUT_DIR = os.path.join(RDS, 'output')
REPO_DIR = os.path.join(RDS, 'largeliterarymodels')
MODEL = 'Qwen/Qwen3.6-27B'
WORKERS = 4

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Texts:  {len(glob.glob(os.path.join(TEXT_DIR, "*.jsonl")))} JSONL files')
print(f'Output: {len(glob.glob(os.path.join(OUTPUT_DIR, "*.json")))} done so far')

## 1. Start vLLM server

In [ ]:
vllm_log = os.path.join(RDS, 'vllm.log')
proc = subprocess.Popen([
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL,
    '--port', '8000',
    '--host', '127.0.0.1',
    '--enable-prefix-caching',
    '--gpu-memory-utilization', '0.90',
    '--max-model-len', '32768',
    '--disable-log-requests',
], stdout=open(vllm_log, 'w'), stderr=subprocess.STDOUT)
print(f'vLLM starting (pid {proc.pid}), logging to {vllm_log}')

In [ ]:
for i in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/health')
        print(f'vLLM ready after {i * 5}s!')
        break
    except:
        if i % 6 == 0:
            print(f'Waiting... ({i * 5}s)')
        time.sleep(5)
else:
    print('vLLM failed to start. Check log:')
    !tail -20 {vllm_log}

## 2. Run batch

In [ ]:
!python {REPO_DIR}/scripts/batch_social_network.py \
    --text-dir {TEXT_DIR} \
    --output-dir {OUTPUT_DIR} \
    --model vllm-qwen36 \
    --workers {WORKERS}

## 3. Check progress

In [ ]:
n_input = len(glob.glob(os.path.join(TEXT_DIR, '*.jsonl')))
n_done = len(glob.glob(os.path.join(OUTPUT_DIR, '*.json')))
print(f'{n_done}/{n_input} texts complete ({n_done/max(1,n_input)*100:.0f}%)')

if n_done > 0:
    latest = max(glob.glob(os.path.join(OUTPUT_DIR, '*.json')), key=os.path.getmtime)
    with open(latest) as f:
        d = json.load(f)
    m = d.get('metadata', {})
    print(f'\nLatest: {os.path.basename(latest)}')
    print(f'  Source: {m.get("source", "?")}')
    print(f'  Chars: {len(d.get("characters", []))}, Events: {len(d.get("events", []))}')
    print(f'  Elapsed: {m.get("elapsed_seconds", 0)/60:.1f} min')

## 4. Cleanup

Stop vLLM when done (or let the job wall time kill it).

In [ ]:
proc.terminate()
print('vLLM stopped')